In [1]:
import pandas as pd
pd.set_option("display.max_columns", None) # display all columns

from pathlib import Path

BASE_DIR = Path.cwd().parent

clean_transactions = BASE_DIR / "resources" / "clean_mpesa_transactions.csv"

df = pd.read_csv(clean_transactions)

In [2]:
print("Columns:", df.columns.tolist())
print("Number of transactions:", len(df))
print("Label counts:", df['Category'].value_counts())

Columns: ['Receipt No', 'Completion Time', 'Details', 'Transaction Status', 'Paid In', 'Withdrawn', 'Balance', 'Notes', 'Year', 'Month', 'Date', 'Day', 'Amount', 'Category']
Number of transactions: 440
Label counts: Category
Expense    323
Income     103
Neutral     14
Name: count, dtype: int64


## Clustering to reveal transaction patterns

In [3]:
from sklearn.cluster import DBSCAN
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight embedding model
embeddings = model.encode(df['Details'].tolist())

embeddings[:5]

array([[-0.0518424 ,  0.05095714, -0.00100877, ...,  0.02403557,
        -0.05793547, -0.0328618 ],
       [-0.04160628,  0.06078717,  0.0122063 , ..., -0.04113406,
         0.01323489, -0.06800348],
       [-0.04161704,  0.07661207, -0.0026605 , ..., -0.05808166,
        -0.01867538, -0.05602515],
       [-0.04161704,  0.07661207, -0.0026605 , ..., -0.05808166,
        -0.01867538, -0.05602515],
       [-0.0518424 ,  0.05095714, -0.00100877, ...,  0.02403557,
        -0.05793547, -0.0328618 ]], dtype=float32)

In [4]:
clustering = DBSCAN(eps=0.2, min_samples=1, metric='cosine').fit(embeddings)
df['Cluster'] = clustering.labels_

In [6]:
# Grouping by cluster to inspect patterns
clusters = df.groupby('Cluster')['Details'].apply(list)
sorted_clusters = clusters.sort_values(key=lambda x: x.map(len), ascending=False)

In [ ]:
print("Clustered Patterns:")
for cluster_id, messages in sorted_clusters.items():
    print(f"\nCluster {cluster_id} (n={len(messages)}):")
    for msg in messages[:5]:  # showing first 5 messages per cluster
        print(f"  {msg}")

### Classification Stage 1: Regex

In [7]:
import re

def classify_with_regex(text):
    text = text.lower()
    
    patterns = {
        r"funds received from": "Income from Mobile Money",
        r"business payment from": "Income from Bank",
        r"deposit of funds at agent till": "Deposited to M-PESA",
        r"customer transfer to": "Mobile Money Transfer",
        r"customer transfer of funds charge": "Transaction Cost",
        r"merchant payment": "Shopping",
        r"airtel money": "Mobile Money Transfer",
        r"offnet c2b transfer": "Mobile Money Transfer",
        r"kplc|water": "Utility Bill",
        r"airtime purchase": "Airtime Purchase",
        r"bundle purchase": "Internet Bundles Buy",
        r"pay bill charge": "Transaction Cost",
        r"pay bill": "Paybill Payment",
        r"withdrawal charge": "Transaction Cost",
        r"customer withdrawal at agent till": "Withdrawn from MPESA",
        r"customer payment to small business": "Shopping",
        r"pay bill online": "Online Bill",
        r"supermarket|naivas|carrefour|shop|store|mart": "Shopping",
        r"uber|bolt|taxi": "Transport",
    }
    
    for pattern, label in patterns.items():
        if re.search(pattern, text):
            return label
    
    return None

df["Subcategory"] = df["Details"].apply(lambda x: classify_with_regex(x))

In [8]:
# Transactions not classified by the regex rules
ml_data = df[df["Subcategory"].isna()]
ml_data.shape

(0, 16)

## Full classification function (Regex + LLM fallback)

In [9]:
from groq import Groq
import os

from dotenv import load_dotenv
load_dotenv()

groq = Groq()

def classify_with_llm(text):
    prompt = f"""
    You are a financial transaction classifier. Classify this transaction into one of the following categories:

    Income from Mobile Money, Income from Bank, Deposited to M-PESA, Mobile Money Transfer,
    Transaction Cost, Shopping, Airtime Purchase, Internet Bundles Buy, Paybill Payment,
    Withdrawn from MPESA, Online Bill, Utility Bill, Transport

    Transaction:
    {text}

    Return only the category inside <category></category>.
    Do not include any explanation, only the category tag.
    """
    chat_completion = groq.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.3-70b-versatile",
        temperature=0.3
    )
    content = chat_completion.choices[0].message.content
    match = re.search(r'<category>(.*)<\/category>', content, flags=re.DOTALL)
    category = "Unclassified"
    if match:
        category = match.group(1).strip()
    return category


## Applying classification to full CSV

In [10]:
def classify_transaction(text):
    label = classify_with_regex(text)
    if label:
        return label, "regex"

    # LLM fallback
    llm_label = classify_with_llm(text)
    return llm_label, "llm"

In [11]:
results = df["Details"].apply(classify_transaction)
df["Predicted_Label"] = results.apply(lambda x: x[0])
df["Method"] = results.apply(lambda x: x[1])

In [12]:
# Saving the results
classified_transactions = BASE_DIR / "resources" / "mpesa_classified.csv"

df.to_csv(classified_transactions, index=False)

## Insights

In [ ]:
df["Predicted_Label"].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.countplot(x="Predicted_Label", data=df, order=df["Predicted_Label"].value_counts().index)
plt.xticks(rotation=45)
plt.show()